In [ ]:
"""ResNet in PyTorch.
ImageNet-Style ResNet
[1] Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun
    Deep Residual Learning for Image Recognition. arXiv:1512.03385
Adapted from: https://github.com/bearpaw/pytorch-classification
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, is_last=False):
        super(BasicBlock, self).__init__()
        self.is_last = is_last
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        preact = out
        out = F.relu(out)
        if self.is_last:
            return out, preact
        else:
            return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, is_last=False):
        super(Bottleneck, self).__init__()
        self.is_last = is_last
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion * planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion * planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        preact = out
        out = F.relu(out)
        if self.is_last:
            return out, preact
        else:
            return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, in_channel=3, zero_init_residual=False):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(in_channel, 64, kernel_size=3, stride=1, padding=1,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves
        # like an identity. This improves the model by 0.2~0.3% according to:
        # https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for i in range(num_blocks):
            stride = strides[i]
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, layer=100):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return out


def resnet18(**kwargs):
    return ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)


def resnet34(**kwargs):
    return ResNet(BasicBlock, [3, 4, 6, 3], **kwargs)


def resnet50(**kwargs):
    return ResNet(Bottleneck, [3, 4, 6, 3], **kwargs)


def resnet101(**kwargs):
    return ResNet(Bottleneck, [3, 4, 23, 3], **kwargs)


model_dict = {
    'resnet18': [resnet18, 512],
    'resnet34': [resnet34, 512],
    'resnet50': [resnet50, 2048],
    'resnet101': [resnet101, 2048],
}


class LinearBatchNorm(nn.Module):
    """Implements BatchNorm1d by BatchNorm2d, for SyncBN purpose"""
    def __init__(self, dim, affine=True):
        super(LinearBatchNorm, self).__init__()
        self.dim = dim
        self.bn = nn.BatchNorm2d(dim, affine=affine)

    def forward(self, x):
        x = x.view(-1, self.dim, 1, 1)
        x = self.bn(x)
        x = x.view(-1, self.dim)
        return x




class ResNet_Model(nn.Module):
    """encoder + classifier"""
    def __init__(self, name='resnet50', num_classes=10):
        super(ResNet_Model, self).__init__()
        model_fun, dim_in = model_dict[name]
        self.encoder = model_fun()
        self.trans = nn.Linear(dim_in, 768)
        # self.fc = nn.Linear(dim_in, num_classes)
        # self.logistic_regression = nn.Linear(1, 1)

    def forward(self, x):
        return self.trans(self.encoder(x))

    def forward_repre(self, x):
        encoded = self.encoder(x)
        encoded = self.trans(encoded)
        return encoded#, self.fc(encoded)


class LinearClassifier(nn.Module):
    """Linear classifier"""
    def __init__(self, name='resnet50', num_classes=10):
        super(LinearClassifier, self).__init__()
        _, feat_dim = model_dict[name]
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, features):
        return self.fc(features)

In [ ]:
class PartialDataset(torch.utils.data.Dataset):
    def __init__(self, parent_ds, offset, length):
        self.parent_ds = parent_ds
        self.offset = offset
        self.length = length
        assert len(parent_ds) >= offset + length, Exception("Parent Dataset not long enough")
        super(PartialDataset, self).__init__()

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        return self.parent_ds[i + self.offset]


def validation_split(dataset, val_share=0.1):
    """
       Split a (training and vaidation combined) dataset into training and validation.
       Note that to be statistically sound, the items in the dataset should be statistically
       independent (e.g. not sorted by class, not several instances of the same dataset that
       could end up in either set).
       inputs:
          dataset:   ("training") dataset to split into training and validation
          val_share: fraction of validation data (should be 0<val_share<1, default: 0.1)
       returns: input dataset split into test_ds, val_ds
    """
    val_offset = int(len(dataset) * (1 - val_share))
    return PartialDataset(dataset, 0, val_offset), PartialDataset(dataset, val_offset, len(dataset) - val_offset)


In [ ]:
class RandomImages50k(torch.utils.data.Dataset):

    def __init__(self, transform=None, exclude_cifar=False):

        data_file = np.load('/nobackup-slow/dataset/my_xfdu/300K_random_images.npy')
        indices = np.random.permutation(len(data_file))
        self.data_file = data_file[indices[:50100]]
        self.offset = 0     # offset index

        self.transform = transform
        self.exclude_cifar = exclude_cifar

    def __getitem__(self, index):
        index = (index + self.offset) % 50099

        if self.exclude_cifar:
            while self.in_cifar(index):
                index = np.random.randint(50100)


        data = self.data_file[index]
        img = np.asarray(data, dtype='uint8')


        if self.transform is not None:
            img = self.transform(img)

        return img, 0  # 0 is the class

    def __len__(self):
        return 50100

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import os
import time
import torch
import torch.backends.cudnn as cudnn
import torchvision.transforms as trn
import torchvision.datasets as dset
import torch.nn.functional as F

# from resnet_anchor import ResNet_Model
# from utils.validation_dataset import validation_split
# from utils.out_dataset import RandomImages50k

# Manually replacing argparse values
dataset = 'cifar100'
model = 'wrn'
epochs = 50
learning_rate = 0.1
batch_size = 256
test_bs = 200
momentum = 0.9
decay = 0.0005
save = './snapshots/'
load = ''
test = False
ngpu = 1
prefetch = 4
my_info = 'PND'
seed = 1

save_info = 'pnd_cifar100'
save = os.path.join(save, save_info)
if not os.path.isdir(save): os.makedirs(save)

state = {
    'dataset': dataset,
    'model': model,
    'epochs': epochs,
    'learning_rate': learning_rate,
    'batch_size': batch_size,
    'test_bs': test_bs,
    'momentum': momentum,
    'decay': decay,
    'save': save,
    'load': load,
    'test': test,
    'ngpu': ngpu,
    'prefetch': prefetch,
    'my_info': my_info,
    'seed': seed
}
print(state)

# Seed
torch.manual_seed(1)
np.random.seed(seed)

# Normalization for CIFAR
mean = [x / 255 for x in [125.3, 123.0, 113.9]]
std = [x / 255 for x in [63.0, 62.1, 66.7]]
train_transform = trn.Compose([trn.RandomHorizontalFlip(), trn.RandomCrop(32, padding=4),
                               trn.ToTensor(), trn.Normalize(mean, std)])
test_transform = trn.Compose([trn.ToTensor(), trn.Normalize(mean, std)])

# Load CIFAR-100
dataset_path = '/nobackup-slow/dataset/my_xfdu/cifarpy'
train_data_in = dset.CIFAR100(dataset_path, train=True, transform=train_transform, download = True)
test_data = dset.CIFAR100(dataset_path, train=False, transform=test_transform , download = True)
num_classes = 100

anchor = torch.from_numpy(np.load('token_embed_c100.npy')).float().cuda()
anchor = F.normalize(anchor, dim=1)

train_loader_in = torch.utils.data.DataLoader(train_data_in, batch_size=batch_size,
                                               shuffle=True, num_workers=prefetch, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size,
                                          shuffle=False, num_workers=prefetch, pin_memory=True)

# Model
net = ResNet_Model(name='resnet34', num_classes=num_classes)
if load and os.path.isfile(load):
    net.load_state_dict(torch.load(load))
    print('Model restored from', load)

if ngpu > 1:
    net = torch.nn.DataParallel(net, device_ids=list(range(ngpu)))
if ngpu > 0:
    net.cuda()
    torch.cuda.manual_seed(1)
cudnn.benchmark = True

optimizer = torch.optim.SGD(net.parameters(), lr=state['learning_rate'], momentum=state['momentum'],
                            weight_decay=state['decay'], nesterov=True)

def cosine_annealing(step, total_steps, lr_max, lr_min):
    return lr_min + (lr_max - lr_min) * 0.5 * (1 + np.cos(step / total_steps * np.pi))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_annealing(
        step,
        epochs * len(train_loader_in),
        1,
        1e-6 / learning_rate))

# ---------- PND LOSS ----------
def projected_normal_logpdf(features, class_means, temperature=0.1):
    features = F.normalize(features, dim=1)
    class_means = F.normalize(class_means, dim=1)
    logits = torch.matmul(features, class_means.T)
    return logits / temperature

# Training loop with PND
def train_pnd():
    net.train()
    loss_avg = 0.0
    for _, in_set in enumerate(train_loader_in):
        data, target = in_set[0].cuda(), in_set[1].cuda()
        features = net(data)
        logits = projected_normal_logpdf(features, anchor, temperature=0.1)
        loss = F.cross_entropy(logits, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        loss_avg = loss_avg * 0.8 + float(loss) * 0.2
    print(scheduler.get_last_lr())
    state['train_loss'] = loss_avg

# Save info
results_path = os.path.join(save, dataset + '_' + model + '_s' + str(seed) + '_' + my_info + '_training_results.csv')
with open(results_path, 'w') as f:
    f.write('epoch,time(s),train_loss\n')

print('Beginning Training\n')

# Main loop
for epoch in range(0, epochs):
    state['epoch'] = epoch
    begin_epoch = time.time()

    train_pnd()

    model_path = os.path.join(save, dataset + '_' + model + '_s' + str(seed) + '_' + my_info + '_epoch_' + str(epoch) + '.pt')
    torch.save(net.state_dict(), model_path)

    prev_path = os.path.join(save, dataset + '_' + model + '_s' + str(seed) + '_' + my_info + '_epoch_' + str(epoch - 1) + '.pt')
    if os.path.exists(prev_path): os.remove(prev_path)

    with open(results_path, 'a') as f:
        f.write('%03d,%05d,%0.6f\n' % ((epoch + 1), time.time() - begin_epoch, state['train_loss']))

    print('Epoch {0:3d} | Time {1:5d} | Train Loss {2:.4f}'.format(
        (epoch + 1), int(time.time() - begin_epoch), state['train_loss']))

# Save extracted ID features
number_dict = {i: 0 for i in range(num_classes)}
data_dict = torch.zeros(num_classes, 500, 768).cuda()

net.eval()
with torch.no_grad():
    for _, in_set in enumerate(train_loader_in):
        data, target = in_set[0].cuda(), in_set[1].cuda()
        feat = net(data)
        target_numpy = target.cpu().data.numpy()
        for index in range(len(target)):
            dict_key = target_numpy[index]
            if number_dict[dict_key] < 500:
                data_dict[dict_key][number_dict[dict_key]] = feat[index].detach()
                number_dict[dict_key] += 1

np.save('./id_feat_cifar100_pnd.npy', data_dict.cpu().numpy())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

# Load the saved PND features
data_dict = np.load('./id_feat_cifar100_pnd.npy')  # shape: [100, 500, 768]

# Flatten into [100*500, 768] and create labels
features = data_dict.reshape(-1, 768)
labels = np.array([[i] * 500 for i in range(100)]).flatten()

# Run t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
features_2d = tsne.fit_transform(features)

# Plot
plt.figure(figsize=(12, 10))
palette = sns.color_palette("hsv", 100)  # distinct colors for each class

sns.scatterplot(x=features_2d[:, 0], y=features_2d[:, 1], hue=labels, palette=palette, legend=False, s=5)

plt.title("t-SNE of CIFAR-100 PND Features", fontsize=16)
plt.xlabel("t-SNE dim 1")
plt.ylabel("t-SNE dim 2")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import torch
import faiss
import umap
import time
#import matplotlib.pyplot as plt
import faiss.contrib.torch_utils

import torch.nn.functional as F

def KNN_dis_search_decrease(target, index, K=50, select=1,shift=0):
    '''
    data_point: Queue for searching k-th points
    target: the target of the search
    K
    '''
    #Normalize the features

    target_norm = torch.norm(target, p=2, dim=1,  keepdim=True)
    normed_target = target / target_norm
    #start_time = time.time()

    distance, output_index = index.search(normed_target, K)
    k_th_distance = distance[:, -1]
    #k_th_output_index = output_index[:, -1]
    if shift:
        k_th_distance, minD_idx = torch.topk(-k_th_distance, select)
    else:
        k_th_distance, minD_idx = torch.topk(k_th_distance, select)
    #k_th_index = k_th_output_index[minD_idx]
    return minD_idx, k_th_distance

def KNN_dis_search_distance(target, index, K=50, num_points=10, length=2000,depth=342,shift=0):
    '''
    data_point: Queue for searching k-th points
    target: the target of the search
    K
    '''
    #Normalize the features

    target_norm = torch.norm(target, p=2, dim=1,  keepdim=True)
    normed_target = target / target_norm
    #start_time = time.time()

    distance, output_index = index.search(normed_target, K)
    k_th_distance = distance[:, -1]
    k_th = k_th_distance.view(length, -1)
    target_new = target.view(length, -1, depth)
    #k_th_output_index = output_index[:, -1]
    if shift:
        k_th_distance, minD_idx = torch.topk(-k_th, num_points, dim=0)
    else:
        k_th_distance, minD_idx = torch.topk(k_th, num_points, dim=0)
    minD_idx = minD_idx.squeeze()
    point_list = []
    # breakpoint()
    if len(minD_idx.size()) == 1:
        minD_idx = minD_idx.reshape(-1,1)
    for i in range(minD_idx.shape[1]):
        point_list.append(i*length + minD_idx[:,i])
    #return tor+ch.cat(point_list, dim=0)

    return target[torch.cat(point_list)]

def generate_outliers(ID, input_index, negative_samples, ID_points_num=2,
                      K=20, select=1, cov_mat=0.1, sampling_ratio=1.0,
                      pic_nums=30, depth=342, shift=0):
    length = negative_samples.shape[0]
    data_norm = torch.norm(ID, p=2, dim=1, keepdim=True)
    normed_data = ID / data_norm
    rand_ind = np.random.choice(normed_data.shape[0], int(normed_data.shape[0] * sampling_ratio), replace=False)
    index = input_index
    index.add(normed_data[rand_ind])
    minD_idx, k_th = KNN_dis_search_decrease(ID, index, K, select, shift=shift)
    boundary_data = ID[minD_idx]
    # breakpoint()
    minD_idx = minD_idx[np.random.choice(select, int(pic_nums), replace=False)]
    data_point_list = torch.cat([ID[i:i+1].repeat(length,1) for i in minD_idx])
    negative_sample_cov = cov_mat * negative_samples.cuda().repeat(pic_nums,1)
    negative_sample_list = F.normalize(negative_sample_cov + data_point_list, p=2, dim=1)
    # breakpoint()
    point = KNN_dis_search_distance(negative_sample_list, index, K, ID_points_num, length,depth, shift=shift)

    index.reset()

    #return ID[minD_idx]
    return point, boundary_data

def generate_outliers_OOD(ID, input_index, negative_samples, K=100, select=100, sampling_ratio=1.0):
    data_norm = torch.norm(ID, p=2, dim=1, keepdim=True)
    normed_data = ID / data_norm
    rand_ind = np.random.choice(normed_data.shape[1], int(normed_data.shape[1] * sampling_ratio), replace=False)
    index = input_index
    index.add(normed_data[rand_ind])
    minD_idx, k_th = KNN_dis_search_decrease(negative_samples, index, K, select)

    return negative_samples[minD_idx]



def generate_outliers_rand(ID, input_index,
                           negative_samples, ID_points_num=2, K=20, select=1,
                           cov_mat=0.1, sampling_ratio=1.0, pic_nums=10,
                           repeat_times=30, depth=342):
    length = negative_samples.shape[0]
    data_norm = torch.norm(ID, p=2, dim=1, keepdim=True)
    normed_data = ID / data_norm
    rand_ind = np.random.choice(normed_data.shape[1], int(normed_data.shape[1] * sampling_ratio), replace=False)
    index = input_index
    index.add(normed_data[rand_ind])
    minD_idx, k_th = KNN_dis_search_decrease(ID, index, K, select)
    ID_boundary = ID[minD_idx]
    negative_sample_list = []
    for i in range(repeat_times):
        select_idx = np.random.choice(select, int(pic_nums), replace=False)
        sample_list = ID_boundary[select_idx]
        mean = sample_list.mean(0)
        var = torch.cov(sample_list.T)
        var = torch.mm(negative_samples, var)
        trans_samples = mean + var
        negative_sample_list.append(trans_samples)
    negative_sample_list = torch.cat(negative_sample_list, dim=0)
    point = KNN_dis_search_distance(negative_sample_list, index, K, ID_points_num, length,depth)

    index.reset()

    #return ID[minD_idx]
    return point

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import torch
import torch.nn.functional as F
import faiss
# from KNN import generate_outliers

# ----- PND Sampling Utility -----
def sample_from_pnd(mean_vec, num_candidates, sigma):
    """
    Sample from an isotropic Projected Normal Distribution (PND):
    - mean_vec: the learned latent-space mean vector for a class
    - Perturb around this mean with Gaussian noise, then project onto unit sphere
    This generates candidate points in latent space near the learned mean.
    """
    base = torch.randn(num_candidates, mean_vec.shape[0], device=mean_vec.device) * sigma
    base = base + mean_vec.unsqueeze(0)
    return F.normalize(base, p=2, dim=1)

# ----- Manual Argument Replacement -----
args = {
    'shift': 0,  # 0 = generate OOD (outlier), 1 = generate noisy inlier
    'gaussian_mag_ood_det': 0.07,
    'gaussian_mag_ood_gene': 0.01,
    'K_in_knn': 300,
    'ood_det_select': 50,
    'ood_gene_select': 1000,
    'gpu': 0
}

device = torch.device(f'cuda:{args["gpu"]}' if torch.cuda.is_available() else 'cpu')

# ----- Load Latent Embeddings and Compute Class Means -----
data_dict = torch.from_numpy(np.load('./id_feat_cifar100_pnd.npy')).float().to(device)
num_classes, num_feats, dim = data_dict.shape
class_means = data_dict.mean(dim=1)
class_means = F.normalize(class_means, dim=1)

anchor = torch.from_numpy(np.load('./token_embed_c100.npy')).float().to(device)  # optional

# ----- Setup FAISS KNN Index -----
res = faiss.StandardGpuResources()
KNN_index = faiss.GpuIndexFlatL2(res, dim)

ood_samples = []
total_ID = num_classes * num_feats
if data_dict.numel() == total_ID * dim:
    for cls in range(num_classes):
        ID_feats = F.normalize(data_dict[cls], p=2, dim=1)
        KNN_index.add(ID_feats.cpu().numpy())

        if args['shift']:
            sigma = args['gaussian_mag_ood_gene']
            select = args['ood_gene_select']
            ID_pts = 1
            pic_nums = 100
            shift_flag = 1
        else:
            sigma = args['gaussian_mag_ood_det']
            select = args['ood_det_select']
            ID_pts = 2
            pic_nums = 50
            shift_flag = 0

        for _ in range(num_classes):
            negative_samples = sample_from_pnd(class_means[cls], num_candidates=1500, sigma=sigma)

            sample_pts, _ = generate_outliers(
                ID_feats,
                input_index=KNN_index,
                negative_samples=negative_samples,
                ID_points_num=ID_pts,
                K=args['K_in_knn'],
                select=select,
                cov_mat=sigma,
                sampling_ratio=1.0,
                pic_nums=pic_nums,
                depth=dim,
                shift=shift_flag
            )

            cur_samples = sample_pts if _ == 0 else torch.cat([cur_samples, sample_pts], dim=0)

        scaled = cur_samples * class_means[cls].norm()
        ood_samples.append(scaled)

# ----- Save -----
ood_tensor = torch.stack(ood_samples)
mode = 'inlier' if args['shift'] else 'outlier'
filename = f'./cifar100_{mode}_pnd_embed_noise_{sigma}_select_{select}_KNN_{args["K_in_knn"]}.npy'
np.save(filename, ood_tensor.cpu().numpy())
print(f"Saved embeddings to {filename}")


AttributeError: module 'faiss' has no attribute 'StandardGpuResources'

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import torch
import torch.nn.functional as F
import faiss
# from KNN import generate_outliers

# ----- PND Sampling Utility -----
def sample_from_pnd(mean_vec, num_candidates, sigma):
    """
    Sample from an isotropic Projected Normal Distribution (PND):
    - mean_vec: the learned latent-space mean vector for a class
    - Perturb around this mean with Gaussian noise, then project onto unit sphere
    This generates candidate points in latent space near the learned mean.
    """
    base = torch.randn(num_candidates, mean_vec.shape[0], device=mean_vec.device) * sigma
    base = base + mean_vec.unsqueeze(0)
    return F.normalize(base, p=2, dim=1)

# ----- Manual Argument Replacement -----
args = {
    'shift': 0,  # 0 = generate OOD (outlier), 1 = generate noisy inlier
    'gaussian_mag_ood_det': 0.07,
    'gaussian_mag_ood_gene': 0.01,
    'K_in_knn': 300,
    'ood_det_select': 50,
    'ood_gene_select': 1000,
    'gpu': 0
}

device = torch.device(f'cuda:{args["gpu"]}' if torch.cuda.is_available() else 'cpu')

# ----- Load Latent Embeddings and Compute Class Means -----
data_dict = torch.from_numpy(np.load('./id_feat_cifar100_pnd.npy')).float().to(device)
num_classes, num_feats, dim = data_dict.shape
class_means = data_dict.mean(dim=1)
class_means = F.normalize(class_means, dim=1)

anchor = torch.from_numpy(np.load('./token_embed_c100.npy')).float().to(device)  # optional

# ----- Setup FAISS KNN Index (CPU) -----
KNN_index = faiss.IndexFlatL2(dim)  # CPU version of L2 index

ood_samples = []
total_ID = num_classes * num_feats
if data_dict.numel() == total_ID * dim:
    for cls in range(num_classes):
        ID_feats = F.normalize(data_dict[cls], p=2, dim=1)

        # Make sure to move to CPU before passing to FAISS
        ID_feats_cpu = ID_feats.cpu()
        KNN_index.add(ID_feats_cpu.numpy())

        if args['shift']:
            sigma = args['gaussian_mag_ood_gene']
            select = args['ood_gene_select']
            ID_pts = 1
            pic_nums = 100
            shift_flag = 1
        else:
            sigma = args['gaussian_mag_ood_det']
            select = args['ood_det_select']
            ID_pts = 2
            pic_nums = 50
            shift_flag = 0

        for _ in range(num_classes):
            negative_samples = sample_from_pnd(class_means[cls], num_candidates=1500, sigma=sigma)

            # Make sure to move negative_samples to CPU as well
            negative_samples_cpu = negative_samples.cpu()

            sample_pts, _ = generate_outliers(
                ID_feats_cpu,
                input_index=KNN_index,
                negative_samples=negative_samples_cpu,
                ID_points_num=ID_pts,
                K=args['K_in_knn'],
                select=select,
                cov_mat=sigma,
                sampling_ratio=1.0,
                pic_nums=pic_nums,
                depth=dim,
                shift=shift_flag
            )

            cur_samples = sample_pts if _ == 0 else torch.cat([cur_samples, sample_pts], dim=0)

        scaled = cur_samples * class_means[cls].norm()
        ood_samples.append(scaled)

# ----- Save -----
ood_tensor = torch.stack(ood_samples)
mode = 'inlier' if args['shift'] else 'outlier'
filename = f'./cifar100_{mode}_pnd_embed_noise_{sigma}_select_{select}_KNN_{args["K_in_knn"]}.npy'
np.save(filename, ood_tensor.cpu().numpy())
print(f"Saved embeddings to {filename}")


In [ ]:
from google.colab import drive

# Mount your drive and upload the stable diffusion on your drive
drive.mount('/content/drive')

ckpt = "/content/drive/MyDrive/sd-v1-4.ckpt"


In [ ]:
!pip install pytorch-lightning

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import CLIPTokenizer, CLIPTextModel
from copy import deepcopy
from PIL import Image
import os
!pip install omegaconf
from omegaconf import OmegaConf
import argparse
import math

# Utility functions from util.py
def make_beta_schedule(schedule, n_timestep, linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):
    if schedule == "linear":
        betas = (
                torch.linspace(linear_start ** 0.5, linear_end ** 0.5, n_timestep, dtype=torch.float32) ** 2
        )
    else:
        raise ValueError(f"schedule '{schedule}' unknown.")
    return betas.numpy()




def make_ddim_timesteps(ddim_discr_method, num_ddim_timesteps, num_ddpm_timesteps, verbose=True):
    if ddim_discr_method == 'uniform':
        c = num_ddpm_timesteps // num_ddim_timesteps
        ddim_timesteps = np.asarray(list(range(0, num_ddpm_timesteps, c)))
    elif ddim_discr_method == 'quad':
        ddim_timesteps = ((np.linspace(0, np.sqrt(num_ddpm_timesteps * .8), num_ddim_timesteps)) ** 2).astype(int)
    else:
        raise NotImplementedError(f'There is no ddim discretization method called "{ddim_discr_method}"')
    # Create indices for the selected timesteps
    indices = np.arange(num_ddim_timesteps)
    if verbose:
        print(f'Selected timesteps for ddim sampler: {ddim_timesteps}')
        print(f'Corresponding indices: {indices}')
    return indices, ddim_timesteps  # Return both indices and actual timesteps

def make_ddim_sampling_parameters(alphacums, ddim_timesteps, eta, verbose=True):
    # Use the actual DDPM timesteps to select alphas
    alphas = alphacums[ddim_timesteps].astype(np.float32)
    alphas_prev = np.asarray([alphacums[0]] + alphacums[ddim_timesteps[:-1]].tolist()).astype(np.float32)
    sigmas = eta * np.sqrt((1 - alphas_prev) / (1 - alphas) * (1 - alphas / alphas_prev)).astype(np.float32)
    if verbose:
        print(f'Selected alphas for ddim sampler: a_t: {alphas}; a_(t-1): {alphas_prev}')
        print(f'For the chosen value of eta, which is {eta}, '
              f'this results in the following sigma_t schedule for ddim sampler {sigmas}')
    return sigmas, alphas, alphas_prev

class DDIMSampler:
    def _init_(self, model):
        self.model = model
        self.device = model.device
        self.ddpm_num_timesteps = model.num_timesteps
        self.alphas_cumprod = model.alphas_cumprod
        self.alphas_cumprod_prev = model.alphas_cumprod_prev

    def make_schedule(self, ddim_num_steps, ddim_discr_method="uniform", ddim_eta=0.0, verbose=True):
        indices, ddim_timesteps = make_ddim_timesteps(
            ddim_discr_method=ddim_discr_method,
            num_ddim_timesteps=ddim_num_steps,
            num_ddpm_timesteps=self.ddpm_num_timesteps,
            verbose=verbose
        )
        sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(
            alphacums=self.alphas_cumprod.cpu().numpy(),
            ddim_timesteps=ddim_timesteps,  # Use actual DDPM timesteps here
            eta=ddim_eta,
            verbose=verbose
        )
        self.ddim_timesteps = torch.tensor(indices, device=self.device, dtype=torch.long)  # Use indices
        self.ddim_alphas = torch.tensor(alphas, device=self.device)
        self.ddim_alphas_prev = torch.tensor(alphas_prev, device=self.device)
        self.ddim_sigmas = torch.tensor(sigmas, device=self.device)
        self.ddim_sqrt_one_minus_alphas = torch.sqrt(1. - self.ddim_alphas)

    def sample(self, S, batch_size, shape, conditioning, unconditional_guidance_scale=7.5, unconditional_conditioning=None, ddim_discr_method="uniform", eta=0.0):
        self.make_schedule(ddim_num_steps=S, ddim_discr_method=ddim_discr_method, ddim_eta=eta)
        C, H, W = shape
        size = (batch_size, C, H, W)
        img = noise_like(size, self.device, repeat=False)
        for i, step in enumerate(self.ddim_timesteps):
            index = len(self.ddim_timesteps) - i - 1
            ts = torch.full((batch_size,), step, device=self.device, dtype=torch.long)
            img, _ = self.p_sample_ddim(img, conditioning, ts, index, unconditional_guidance_scale, unconditional_conditioning)
        return img

    def p_sample_ddim(self, x, c, t, index, unconditional_guidance_scale, unconditional_conditioning):
        b = x.shape[0]
        if unconditional_conditioning is None or unconditional_guidance_scale == 1.0:
            e_t = self.model.apply_model(x, t, c)
        else:
            x_in = torch.cat([x] * 2)
            t_in = torch.cat([t] * 2)
            c_in = torch.cat([unconditional_conditioning, c])
            e_t_uncond, e_t = self.model.apply_model(x_in, t_in, c_in).chunk(2)
            e_t = e_t_uncond + unconditional_guidance_scale * (e_t - e_t_uncond)
        a_t = extract_into_tensor(self.ddim_alphas, t, x.shape)
        a_prev = extract_into_tensor(self.ddim_alphas_prev, t, x.shape)
        sigma_t = extract_into_tensor(self.ddim_sigmas, t, x.shape)
        sqrt_one_minus_at = extract_into_tensor(self.ddim_sqrt_one_minus_alphas, t, x.shape)
        pred_x0 = (x - sqrt_one_minus_at * e_t) / a_t.sqrt()
        dir_xt = (1. - a_prev - sigma_t**2).sqrt() * e_t
        noise = sigma_t * noise_like(x.shape, self.device, repeat=False)
        x_prev = a_prev.sqrt() * pred_x0 + dir_xt + noise
        return x_prev, pred_x0

# def make_ddim_timesteps(ddim_discr_method, num_ddim_timesteps, num_ddpm_timesteps, verbose=True):
#     if ddim_discr_method == 'uniform':
#         c = num_ddpm_timesteps // num_ddim_timesteps
#         ddim_timesteps = np.asarray(list(range(0, num_ddpm_timesteps, c)))
#     elif ddim_discr_method == 'quad':
#         ddim_timesteps = ((np.linspace(0, np.sqrt(num_ddpm_timesteps * .8), num_ddim_timesteps)) ** 2).astype(int)
#     else:
#         raise NotImplementedError(f'There is no ddim discretization method called "{ddim_discr_method}"')
#     steps_out = ddim_timesteps
#     if verbose:
#         print(f'Selected timesteps for ddim sampler: {steps_out}')
#     return steps_out

# def make_ddim_sampling_parameters(alphacums, ddim_timesteps, eta, verbose=True):
#     alphas = alphacums[ddim_timesteps].astype(np.float32)
#     alphas_prev = np.asarray([alphacums[0]] + alphacums[ddim_timesteps[:-1]].tolist()).astype(np.float32)
#     sigmas = eta * np.sqrt((1 - alphas_prev) / (1 - alphas) * (1 - alphas / alphas_prev)).astype(np.float32)
#     if verbose:
#         print(f'Selected alphas for ddim sampler: a_t: {alphas}; a_(t-1): {alphas_prev}')
#         print(f'For the chosen value of eta, which is {eta}, '
#               f'this results in the following sigma_t schedule for ddim sampler {sigmas}')
#     return sigmas, alphas, alphas_prev

def noise_like(shape, device, repeat=False):
    repeat_noise = lambda: torch.randn((1, *shape[1:]), device=device).repeat(shape[0], *((1,) * (len(shape) - 1)))
    noise = lambda: torch.randn(shape, device=device)
    return repeat_noise() if repeat else noise()

def extract_into_tensor(a, t, x_shape):
    b, *_ = t.shape
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

# U-Net and Decoder from model.py
def get_timestep_embedding(timesteps, embedding_dim):
    assert len(timesteps.shape) == 1
    half_dim = embedding_dim // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)
    emb = emb.to(device=timesteps.device)
    emb = timesteps.float()[:, None] * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if embedding_dim % 2 == 1:  # zero pad
        emb = torch.nn.functional.pad(emb, (0,1,0,0))
    return emb

def nonlinearity(x):
    return x * torch.sigmoid(x)

def Normalize(in_channels, num_groups=32):
    return torch.nn.GroupNorm(num_groups=num_groups, num_channels=in_channels, eps=1e-6, affine=True)

class Upsample(nn.Module):
    def _init_(self, in_channels, with_conv):
        super()._init_()
        self.with_conv = with_conv
        if self.with_conv:
            self.conv = torch.nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = torch.nn.functional.interpolate(x, scale_factor=2.0, mode="nearest")
        if self.with_conv:
            x = self.conv(x)
        return x

class Downsample(nn.Module):
    def _init_(self, in_channels, with_conv):
        super()._init_()
        self.with_conv = with_conv
        if self.with_conv:
            self.conv = torch.nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=2, padding=0)

    def forward(self, x):
        if self.with_conv:
            pad = (0,1,0,1)
            x = torch.nn.functional.pad(x, pad, mode="constant", value=0)
            x = self.conv(x)
        else:
            x = torch.nn.functional.avg_pool2d(x, kernel_size=2, stride=2)
        return x

class ResnetBlock(nn.Module):
    def _init_(self, *, in_channels, out_channels=None, conv_shortcut=False, dropout, temb_channels=512):
        super()._init_()
        self.in_channels = in_channels
        out_channels = in_channels if out_channels is None else out_channels
        self.out_channels = out_channels
        self.use_conv_shortcut = conv_shortcut
        self.norm1 = Normalize(in_channels)
        self.conv1 = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        if temb_channels > 0:
            self.temb_proj = torch.nn.Linear(temb_channels, out_channels)
        self.norm2 = Normalize(out_channels)
        self.dropout = torch.nn.Dropout(dropout)
        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        if self.in_channels != self.out_channels:
            if self.use_conv_shortcut:
                self.conv_shortcut = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
            else:
                self.nin_shortcut = torch.nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x, temb):
        h = x
        h = self.norm1(h)
        h = nonlinearity(h)
        h = self.conv1(h)
        if temb is not None:
            h = h + self.temb_proj(nonlinearity(temb))[:,:,None,None]
        h = self.norm2(h)
        h = nonlinearity(h)
        h = self.dropout(h)
        h = self.conv2(h)
        if self.in_channels != self.out_channels:
            if self.use_conv_shortcut:
                x = self.conv_shortcut(x)
            else:
                x = self.nin_shortcut(x)
        return x + h

class AttnBlock(nn.Module):
    def _init_(self, in_channels):
        super()._init_()
        self.in_channels = in_channels
        self.norm = Normalize(in_channels)
        self.q = torch.nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0)
        self.k = torch.nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0)
        self.v = torch.nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0)
        self.proj_out = torch.nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        h_ = x
        h_ = self.norm(h_)
        q = self.q(h_)
        k = self.k(h_)
        v = self.v(h_)
        b, c, h, w = q.shape
        q = q.reshape(b, c, h*w)
        q = q.permute(0, 2, 1)  # b,hw,c
        k = k.reshape(b, c, h*w)  # b,c,hw
        w_ = torch.bmm(q, k)  # b,hw,hw
        w_ = w_ * (int(c)(-0.5))
        w_ = torch.nn.functional.softmax(w_, dim=2)
        v = v.reshape(b, c, h*w)
        w_ = w_.permute(0, 2, 1)  # b,hw,hw
        h_ = torch.bmm(v, w_)
        h_ = h_.reshape(b, c, h, w)
        h_ = self.proj_out(h_)
        return x + h_

def make_attn(in_channels, attn_type="vanilla"):
    assert attn_type in ["vanilla", "linear", "none"], f'attn_type {attn_type} unknown'
    if attn_type == "vanilla":
        return AttnBlock(in_channels)
    elif attn_type == "none":
        return nn.Identity(in_channels)
    else:
        raise ValueError("LinearAttention not provided")

class UNetModel(nn.Module):
    def _init_(self, *, ch, out_ch, ch_mult=(1,2,4,8), num_res_blocks, attn_resolutions, dropout=0.0,
                 resamp_with_conv=True, in_channels, resolution, use_timestep=True, use_linear_attn=False,
                 attn_type="vanilla"):
        super()._init_()
        if use_linear_attn: attn_type = "linear"
        self.ch = ch
        self.temb_ch = self.ch * 4
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution
        self.in_channels = in_channels
        self.use_timestep = use_timestep
        if self.use_timestep:
            self.temb = nn.Module()
            self.temb.dense = nn.ModuleList([
                torch.nn.Linear(self.ch, self.temb_ch),
                torch.nn.Linear(self.temb_ch, self.temb_ch),
            ])
        self.conv_in = torch.nn.Conv2d(in_channels, self.ch, kernel_size=3, stride=1, padding=1)
        curr_res = resolution
        in_ch_mult = (1,) + tuple(ch_mult)
        self.down = nn.ModuleList()
        for i_level in range(self.num_resolutions):
            block = nn.ModuleList()
            attn = nn.ModuleList()
            block_in = ch * in_ch_mult[i_level]
            block_out = ch * ch_mult[i_level]
            for i_block in range(self.num_res_blocks):
                block.append(ResnetBlock(in_channels=block_in, out_channels=block_out,
                                         temb_channels=self.temb_ch, dropout=dropout))
                block_in = block_out
                if curr_res in attn_resolutions:
                    attn.append(make_attn(block_in, attn_type=attn_type))
            down = nn.Module()
            down.block = block
            down.attn = attn
            if i_level != self.num_resolutions - 1:
                down.downsample = Downsample(block_in, resamp_with_conv)
                curr_res = curr_res // 2
            self.down.append(down)
        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in,
                                       temb_channels=self.temb_ch, dropout=dropout)
        self.mid.attn_1 = make_attn(block_in, attn_type=attn_type)
        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in,
                                       temb_channels=self.temb_ch, dropout=dropout)
        self.up = nn.ModuleList()
        for i_level in reversed(range(self.num_resolutions)):
            block = nn.ModuleList()
            attn = nn.ModuleList()
            block_out = ch * ch_mult[i_level]
            skip_in = ch * ch_mult[i_level]
            for i_block in  range(self.num_res_blocks + 1):
                if i_block == self.num_res_blocks:
                    skip_in = ch * in_ch_mult[i_level]
                block.append(ResnetBlock(in_channels=block_in + skip_in, out_channels=block_out,
                                         temb_channels=self.temb_ch, dropout=dropout))
                block_in = block_out
                if curr_res in attn_resolutions:
                    attn.append(make_attn(block_in, attn_type=attn_type))
            up = nn.Module()
            up.block = block
            up.attn = attn
            if i_level != 0:
                up.upsample = Upsample(block_in, resamp_with_conv)
                curr_res = curr_res * 2
            self.up.insert(0, up)
        self.norm_out = Normalize(block_in)
        self.conv_out = torch.nn.Conv2d(block_in, out_ch, kernel_size=3, stride=1, padding=1)

    def forward(self, x, t=None, context=None):

        if self.use_timestep:
            assert t is not None
            temb = get_timestep_embedding(t, self.ch)
            temb = self.temb.dense[0](temb)
            temb = nonlinearity(temb)
            temb = self.temb.dense[1](temb)
        else:
            temb = None
        hs = [self.conv_in(x)]
        for i_level in range(self.num_resolutions):
            for i_block in range(self.num_res_blocks):
                h = self.down[i_level].block[i_block](hs[-1], temb)
                if len(self.down[i_level].attn) > 0:
                    h = self.down[i_level].attn[i_block](h)
                hs.append(h)
            if i_level != self.num_resolutions - 1:
                hs.append(self.down[i_level].downsample(hs[-1]))
        h = hs[-1]
        h = self.mid.block_1(h, temb)
        h = self.mid.attn_1(h)
        h = self.mid.block_2(h, temb)
        for i_level in reversed(range(self.num_resolutions)):
            for i_block in range(self.num_res_blocks + 1):
                h = self.up[i_level].block[i_block](torch.cat([h, hs.pop()], dim=1), temb)
                if len(self.up[i_level].attn) > 0:
                    h = self.up[i_level].attn[i_block](h)
            if i_level != 0:
                h = self.up[i_level].upsample(h)
        h = self.norm_out(h)
        h = nonlinearity(h)
        h = self.conv_out(h)
        return h

class Decoder(nn.Module):
    def _init_(self, *, ch, out_ch, ch_mult=(1,2,4,8), num_res_blocks, attn_resolutions, dropout=0.0,
                 resamp_with_conv=True, in_channels, resolution, z_channels, give_pre_end=False, tanh_out=False,
                 use_linear_attn=False, attn_type="vanilla"):
        super()._init_()
        if use_linear_attn: attn_type = "linear"
        self.ch = ch
        self.temb_ch = 0
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution
        self.in_channels = in_channels
        self.give_pre_end = give_pre_end
        self.tanh_out = tanh_out
        in_ch_mult = (1,) + tuple(ch_mult)
        block_in = ch * ch_mult[self.num_resolutions - 1]
        curr_res = resolution // 2 ** (self.num_resolutions - 1)
        self.z_shape = (1, z_channels, curr_res, curr_res)
        self.conv_in = torch.nn.Conv2d(z_channels, block_in, kernel_size=3, stride=1, padding=1)
        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in,
                                       temb_channels=self.temb_ch, dropout=dropout)
        self.mid.attn_1 = make_attn(block_in, attn_type=attn_type)
        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in,
                                       temb_channels=self.temb_ch, dropout=dropout)
        self.up = nn.ModuleList()
        for i_level in reversed(range(self.num_resolutions)):
            block = nn.ModuleList()
            attn = nn.ModuleList()
            block_out = ch * ch_mult[i_level]
            for i_block in range(self.num_res_blocks + 1):
                block.append(ResnetBlock(in_channels=block_in, out_channels=block_out,
                                         temb_channels=self.temb_ch, dropout=dropout))
                block_in = block_out
                if curr_res in attn_resolutions:
                    attn.append(make_attn(block_in, attn_type=attn_type))
            up = nn.Module()
            up.block = block
            up.attn = attn
            if i_level != 0:
                up.upsample = Upsample(block_in, resamp_with_conv)
                curr_res = curr_res * 2
            self.up.insert(0, up)
        self.norm_out = Normalize(block_in)
        self.conv_out = torch.nn.Conv2d(block_in, out_ch, kernel_size=3, stride=1, padding=1)

    def forward(self, z):
        self.last_z_shape = z.shape
        temb = None
        h = self.conv_in(z)
        h = self.mid.block_1(h, temb)
        h = self.mid.attn_1(h)
        h = self.mid.block_2(h, temb)
        for i_level in reversed(range(self.num_resolutions)):
            for i_block in range(self.num_res_blocks + 1):
                h = self.up[i_level].block[i_block](h, temb)
                if len(self.up[i_level].attn) > 0:
                    h = self.up[i_level].attn[i_block](h)
            if i_level != 0:
                h = self.up[i_level].upsample(h)
        if self.give_pre_end:
            return h
        h = self.norm_out(h)
        h = nonlinearity(h)
        h = self.conv_out(h)
        if self.tanh_out:
            h = torch.tanh(h)
        return h

# Simplified Latent Diffusion Model
class LatentDiffusion(torch.nn.Module):
    def _init_(self, config_path, ckpt_path):
        super()._init_()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        config = OmegaConf.load(config_path)
        # Initialize diffusion parameters
        self.num_timesteps = config.model.params.timesteps
        betas = make_beta_schedule(
            schedule="linear",
            n_timestep=config.model.params.timesteps,
            linear_start=config.model.params.linear_start,
            linear_end=config.model.params.linear_end
        )
        self.register_buffer("betas", torch.from_numpy(betas).float())
        self.register_buffer("alphas_cumprod", torch.from_numpy(np.cumprod(1. - betas, axis=0)).float())
        self.register_buffer("alphas_cumprod_prev", torch.from_numpy(np.append(1., np.cumprod(1. - betas, axis=0)[:-1])).float())
        self.register_buffer("sqrt_alphas_cumprod", torch.sqrt(self.alphas_cumprod))
        self.register_buffer("sqrt_one_minus_alphas_cumprod", torch.sqrt(1. - self.alphas_cumprod))
        # Initialize U-Net
        unet_config = config.model.params.unet_config.params
        self.diffusion_model = UNetModel(
            ch=unet_config.model_channels,
            out_ch=unet_config.out_channels,
            ch_mult=unet_config.channel_mult,
            num_res_blocks=unet_config.num_res_blocks,
            attn_resolutions=unet_config.attention_resolutions,
            dropout=0.0,
            resamp_with_conv=True,
            in_channels=unet_config.in_channels,
            resolution=unet_config.image_size,
            use_timestep=True,
            use_linear_attn=unet_config.get("use_linear_attn", False),
            attn_type="vanilla"
        )
        # Initialize conditioning model
        self.cond_stage_model = FrozenCLIPEmbedder(device=self.device)
        # Initialize first-stage model
        first_stage_config = config.model.params.first_stage_config.params
        self.first_stage_model = VQModelInterface(first_stage_config)
        self.scale_factor = config.model.params.scale_factor
        # Load checkpoint
        self.load_checkpoint(ckpt_path)
        self.to(self.device)
        self.eval()

    def load_checkpoint(self, ckpt_path):
        if not os.path.exists(ckpt_path):
            raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"]

        self.load_state_dict(sd, strict=False)
        print(f"Loaded checkpoint from {ckpt_path}")

    def get_learned_conditioning(self, text, class_index, opt):

        c = self.cond_stage_model.encode(text, class_index, opt)

        return c

    def apply_model(self, x, t, c):

        print("Hello 418")
        print("Conditioning c:", type(c), c.shape if isinstance(c, torch.Tensor) else c.keys())

        if isinstance(c, torch.Tensor):
            c = {"c_crossattn": [c]}


        print([x.shape if isinstance(x, torch.Tensor) else type(x) for x in c["c_crossattn"]])


        cc = torch.cat(c["c_crossattn"], 1)
        out = self.diffusion_model(x, t, context=cc)
        return out

    def decode_first_stage(self, z):
        z = 1. / self.scale_factor * z
        return self.first_stage_model.decode(z, force_not_quantize=True)

# Simplified FrozenCLIPEmbedder
class FrozenCLIPEmbedder(torch.nn.Module):
    def _init_(self, version="openai/clip-vit-large-patch14", device="cuda", max_length=77):
        super()._init_()
        self.tokenizer = CLIPTokenizer.from_pretrained(version)
        self.transformer = CLIPTextModel.from_pretrained(version)
        self.device = device
        self.max_length = max_length
        self.freeze()

    def freeze(self):
        self.transformer = self.transformer.eval()
        for param in self.parameters():
            param.requires_grad = False

    def encode(self, text, class_index, opt):
        self.id_data = opt.id_data
        self.outlier_embedding = torch.from_numpy(np.load(opt.loaded_embedding)).to(self.device)
        return self.forward(text, class_index)

    def forward(self, text, class_index):
        fine_labels = [
            "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle", "bicycle", "bottle",
            "bowl", "boy", "bridge", "bus", "butterfly", "camel", "can", "castle", "caterpillar", "cattle",
            "chair", "chimpanzee", "clock", "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
            "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster", "house", "kangaroo", "keyboard",
            "lamp", "lawn_mower", "leopard", "lion", "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain",
            "mouse", "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear", "pickup_truck", "pine_tree",
            "plain", "plate", "poppy", "porcupine", "possum", "rabbit", "raccoon", "ray", "road", "rocket",
            "rose", "sea", "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
            "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank", "telephone", "television", "tiger", "tractor",
            "train", "trout", "tulip", "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm"
        ]
        if text[0] != '':
            tmp_token = self.tokenizer([fine_labels[class_index]], truncation=True, max_length=self.max_length,
                                       return_length=True, return_overflowing_tokens=False, padding="max_length",
                                       return_tensors="pt")
            tokens = tmp_token["input_ids"].to(self.device)
            original_embed = deepcopy(self.transformer.text_model.embeddings.token_embedding.weight[tokens[0][1]])
            original_id = tokens[0][1]
            outlier = self.outlier_embedding[class_index][np.random.choice(10, 1)[0]]
            self.transformer.text_model.embeddings.token_embedding.weight[original_id] = outlier
        batch_encoding = self.tokenizer(text, truncation=True, max_length=self.max_length, return_length=True,
                                        return_overflowing_tokens=False, padding="max_length", return_tensors="pt")
        tokens = batch_encoding["input_ids"].to(self.device)
        outputs = self.transformer(input_ids=tokens)
        z = outputs.last_hidden_state
        if text[0] != '':
            self.transformer.text_model.embeddings.token_embedding.weight[original_id] = original_embed
        return z

# Simplified VQModelInterface
class VQModelInterface(torch.nn.Module):
    def _init_(self, config):
        super()._init_()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.embed_dim = config.embed_dim
        ddconfig = config.ddconfig
        self.decoder = Decoder(
            ch=ddconfig.ch,
            out_ch=ddconfig.out_ch,
            ch_mult=ddconfig.ch_mult,
            num_res_blocks=ddconfig.num_res_blocks,
            attn_resolutions=ddconfig.attn_resolutions,
            dropout=ddconfig.dropout,
            resamp_with_conv=True,
            in_channels=ddconfig.in_channels,
            resolution=ddconfig.resolution,
            z_channels=ddconfig.z_channels,
            give_pre_end=False,
            tanh_out=False,
            use_linear_attn=False,
            attn_type="vanilla"
        )
        self.post_quant_conv = torch.nn.Conv2d(self.embed_dim, ddconfig.z_channels, 1)
        # Load weights (assumed to be part of checkpoint)
        self.to(self.device)
        self.eval()

    def decode(self, h, force_not_quantize=False):
        quant = h if force_not_quantize else h  # Quantization bypassed
        quant = self.post_quant_conv(quant)
        dec = self.decoder(quant)
        return dec

# Simplified DDIMSampler
# class DDIMSampler:
#     def _init_(self, model):
#         self.model = model
#         self.device = model.device
#         self.ddpm_num_timesteps = model.num_timesteps
#         self.alphas_cumprod = model.alphas_cumprod
#         self.alphas_cumprod_prev = model.alphas_cumprod_prev

#     def make_schedule(self, ddim_num_steps, ddim_discr_method="uniform", ddim_eta=0.0, verbose=True):
#         ddim_timesteps = make_ddim_timesteps(
#             ddim_discr_method=ddim_discr_method,
#             num_ddim_timesteps=ddim_num_steps,
#             num_ddpm_timesteps=self.ddpm_num_timesteps,
#             verbose=verbose
#         )
#         sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(
#             alphacums=self.alphas_cumprod.cpu().numpy(),
#             ddim_timesteps=ddim_timesteps,
#             eta=ddim_eta,
#             verbose=verbose
#         )
#         self.ddim_timesteps = torch.tensor(ddim_timesteps, device=self.device, dtype=torch.long)
#         self.ddim_alphas = torch.tensor(alphas, device=self.device)
#         self.ddim_alphas_prev = torch.tensor(alphas_prev, device=self.device)
#         self.ddim_sigmas = torch.tensor(sigmas, device=self.device)
#         self.ddim_sqrt_one_minus_alphas = torch.sqrt(1. - self.ddim_alphas)

#     def sample(self, S, batch_size, shape, conditioning, unconditional_guidance_scale=7.5, unconditional_conditioning=None, ddim_discr_method="uniform", eta=0.0):
#         self.make_schedule(ddim_num_steps=S, ddim_discr_method=ddim_discr_method, ddim_eta=eta)
#         C, H, W = shape
#         size = (batch_size, C, H, W)
#         img = noise_like(size, self.device, repeat=False)
#         for i, step in enumerate(self.ddim_timesteps):
#             index = len(self.ddim_timesteps) - i - 1
#             ts = torch.full((batch_size,), step, device=self.device, dtype=torch.long)
#             img, _ = self.p_sample_ddim(img, conditioning, ts, index, unconditional_guidance_scale, unconditional_conditioning)
#         return img

#     def p_sample_ddim(self, x, c, t, index, unconditional_guidance_scale, unconditional_conditioning):
#         b = x.shape[0]
#         if unconditional_conditioning is None or unconditional_guidance_scale == 1.0:
#             e_t = self.model.apply_model(x, t, c)
#         else:
#             x_in = torch.cat([x] * 2)
#             t_in = torch.cat([t] * 2)
#             c_in = torch.cat([unconditional_conditioning, c])
#             e_t_uncond, e_t = self.model.apply_model(x_in, t_in, c_in).chunk(2)
#             e_t = e_t_uncond + unconditional_guidance_scale * (e_t - e_t_uncond)
#         a_t = extract_into_tensor(self.ddim_alphas, t, x.shape)
#         a_prev = extract_into_tensor(self.ddim_alphas_prev, t, x.shape)
#         sigma_t = extract_into_tensor(self.ddim_sigmas, t, x.shape)
#         sqrt_one_minus_at = extract_into_tensor(self.ddim_sqrt_one_minus_alphas, t, x.shape)
#         pred_x0 = (x - sqrt_one_minus_at * e_t) / a_t.sqrt()
#         dir_xt = (1. - a_prev - sigma_t**2).sqrt() * e_t
#         noise = sigma_t * noise_like(x.shape, self.device, repeat=False)
#         x_prev = a_prev.sqrt() * pred_x0 + dir_xt + noise
#         return x_prev, pred_x0

# Main function
def main():
    from types import SimpleNamespace

    opt = SimpleNamespace(
        loaded_embedding="/content/Outlier_embeddings.npy",  # Update this if also in Drive
        outdir="/content/outputs/cifar100_ood_samples",
        class_name="apple",
        n_samples=5,
        ddim_steps=5,
        scale=7.5,
        ddim_eta=0.0,
        H=512,
        W=512,
        gaussian_scale=0.0,
        ckpt="/content/drive/MyDrive/sd-v1-4.ckpt",
        config="/content/v1-inference.yaml",  # Update if also in Drive
        id_data="cifar100"
)


    # Validate inputs
    if not os.path.exists(opt.loaded_embedding):
        raise FileNotFoundError(f"Embedding file not found at {opt.loaded_embedding}")
    if not os.path.exists(opt.config):
        raise FileNotFoundError(f"Config file not found at {opt.config}")

    # Initialize model
    model = LatentDiffusion(opt.config, opt.ckpt)

    # CIFAR-100 class names
    class_names = [
        "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle", "bicycle", "bottle",
        "bowl", "boy", "bridge", "bus", "butterfly", "camel", "can", "castle", "caterpillar", "cattle",
        "chair", "chimpanzee", "clock", "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
        "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster", "house", "kangaroo", "keyboard",
        "lamp", "lawn_mower", "leopard", "lion", "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain",
        "mouse", "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear", "pickup_truck", "pine_tree",
        "plain", "plate", "poppy", "porcupine", "possum", "rabbit", "raccoon", "ray", "road", "rocket",
        "rose", "sea", "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
        "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank", "telephone", "television", "tiger", "tractor",
        "train", "trout", "tulip", "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm"
    ]

    # Validate class name
    if opt.class_name not in class_names:
        raise ValueError(f"Class name '{opt.class_name}' not in CIFAR-100 classes")

    # Setup output directory
    os.makedirs(opt.outdir, exist_ok=True)
    class_dir = os.path.join(opt.outdir, str(class_names.index(opt.class_name)))
    os.makedirs(class_dir, exist_ok=True)

    # Generate prompt
    prompt = f"A simple image of the {opt.class_name}"
    prompts = [prompt] * opt.n_samples
    class_index = class_names.index(opt.class_name)

    # Get conditioning embedding
    with torch.no_grad():
        c = model.get_learned_conditioning(prompts, class_index, opt)
        if opt.gaussian_scale > 0:
            c += opt.gaussian_scale * torch.randn_like(c)
        uc = model.get_learned_conditioning([""] * opt.n_samples, class_index, opt)

    # Setup sampler
    sampler = DDIMSampler(model)
    shape = (4, opt.H // 8, opt.W // 8)  # Latent shape (downsampled by f=8)
    with torch.no_grad():
        samples_ddim = sampler.sample(
            S=opt.ddim_steps,
            batch_size=opt.n_samples,
            shape=shape,
            conditioning=c,
            unconditional_guidance_scale=opt.scale,
            unconditional_conditioning=uc,
            ddim_discr_method="uniform",
            eta=opt.ddim_eta
        )

    # Decode and save images
    with torch.no_grad():
        x_samples_ddim = model.decode_first_stage(samples_ddim)
        x_samples_ddim = torch.clamp((x_samples_ddim + 1.0) / 2.0, min=0.0, max=1.0)
        for i, x_sample in enumerate(x_samples_ddim):
            x_sample = 255. * x_sample.permute(1, 2, 0).cpu().numpy()
            img = Image.fromarray(x_sample.astype(np.uint8))
            img.save(os.path.join(class_dir, f"{opt.class_name}_{i:05d}.png"))

if _name_ == "_main_":
    main()